<a href="https://colab.research.google.com/github/robertcchen8615/gemini-cli/blob/main/%E3%80%8Cai_%E5%A0%B1%E5%83%B9%E5%8A%A9%E7%90%86_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AI 報價助理：Prompt-driven 需求規格書互動教學

我們將一步一步完成：

1. 案例背景整理  
2. 痛點分析  
3. 利害關係人分析  
4. 使用情境與 User Story  
5. 專案目標與非目標  
6. 功能需求  
7. 非功能需求  
8. 資料需求  
9. 系統邊界與限制  
10. 驗收標準與 KPI  
11. 整併成完整需求規格書  

---

### 思路
1. 先執行安裝與啟動 Ollama 的區塊  
2. 輸入要使用的模型名稱  
3. 啟動 Gradio 互動介面  
4. 依序完成每個階段  
5. 最後匯出 Markdown 規格書


In [ ]:

# 安裝必要套件
!pip -q install gradio requests pandas


In [ ]:

# 在 Colab Linux 環境安裝 Ollama
# 來源：官方 Linux 安裝方式為 curl -fsSL https://ollama.com/install.sh | sh
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,486 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,962 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd6

In [ ]:

# 啟動 Ollama 服務
import subprocess, time, os, signal, textwrap

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)
print("Ollama server started (or starting). PID:", ollama_proc.pid)


Ollama server started (or starting). PID: 2801


In [ ]:

# 設定模型名稱
MODEL_NAME = "qwen2.5:7b"  # 可自行改成你已經 pull 過的模型
print("Current model:", MODEL_NAME)


Current model: qwen2.5:7b


In [ ]:

# 下載模型（第一次執行需要一些時間）
# 若你要改模型，請先修改上一格的 MODEL_NAME
import subprocess, shlex
subprocess.run(["ollama", "pull", MODEL_NAME], check=False)


CompletedProcess(args=['ollama', 'pull', 'qwen2.5:7b'], returncode=0)

In [ ]:

# 基本 API 測試
import requests, json

base_url = "http://127.0.0.1:11434/api/chat"
payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "你是一位專業的企業系統分析師。"},
        {"role": "user", "content": "請用一句話介紹你自己。"}
    ],
    "stream": False
}
resp = requests.post(base_url, json=payload, timeout=180)
resp.raise_for_status()
data = resp.json()
print(data["message"]["content"])


我是一名专业的企业系统分析师，擅长通过深入业务理解与数据分析，设计优化企业信息系统和流程的专业人士。


In [ ]:

# ===== Prompt 套組與流程設定 =====

from copy import deepcopy

STAGE_DEFS = [
    {
        "id": "stage1_background",
        "title": "1. 背景整理",
        "role": "企業系統分析師",
        "instruction": """請根據使用者提供的企業案例故事，整理成需求規格書前言內容。""",
        "output_requirements": [
            "專案背景",
            "專案動機",
            "現行流程摘要"
        ],
        "template": """你現在是一位{role}。

我正在規劃一個生成式 AI 專案，請根據以下企業案例故事，幫我整理成需求規格書前言用的內容。

請輸出以下三個部分：
1. 專案背景
2. 專案動機
3. 現行流程摘要

要求：
- 用正式、清楚、客觀的語氣
- 不要寫成行銷文案
- 內容要適合放進需求規格書
- 請使用繁體中文
- 請用 Markdown 格式輸出

企業案例故事如下：
{context}
"""
    },
    {
        "id": "stage2_painpoints",
        "title": "2. 痛點分析",
        "role": "企業流程分析顧問",
        "instruction": "從現況流程與背景中整理可被系統解決的痛點。",
        "output_requirements": [
            "主要痛點列表",
            "每個痛點造成的後果",
            "適合用 AI 解決的痛點",
            "比較適合用流程、規則或系統整合解決的痛點"
        ],
        "template": """你現在是一位{role}。

請根據以下企業案例與現況流程，協助我整理痛點分析。

請輸出：
1. 主要痛點列表
2. 每個痛點造成的後果
3. 哪些痛點適合用 AI 解決
4. 哪些痛點比較適合用流程改善、規則或系統整合解決

要求：
- 至少列出 5 個痛點
- 每個痛點都要包含具體說明，不要只寫「效率差」
- 請區分「問題現象」與「真正原因」
- 使用繁體中文
- 用表格輸出

資料如下：
{context}
"""
    },
    {
        "id": "stage3_stakeholders",
        "title": "3. 利害關係人分析",
        "role": "產品經理",
        "instruction": "定義利害關係人、主要使用者與次要使用者。",
        "output_requirements": [
            "利害關係人列表",
            "每個角色的職責",
            "與系統的關係",
            "最在意的事情",
            "主要 / 次要使用者"
        ],
        "template": """你現在是一位{role}。

請根據以下企業案例，幫我分析這個專案的利害關係人與使用者角色。

請輸出：
1. 利害關係人列表
2. 每個角色的職責
3. 每個角色與系統的關係
4. 每個角色最在意的事情
5. 誰是主要使用者，誰是次要使用者

要求：
- 至少列出 4 種角色
- 請用表格輸出
- 語氣要正式，適合放進需求規格書

企業資料如下：
{context}
"""
    },
    {
        "id": "stage4_scenarios",
        "title": "4. 使用情境與 User Story",
        "role": "UX 與需求分析顧問",
        "instruction": "把需求從抽象概念轉成真實使用場景。",
        "output_requirements": [
            "3 到 5 個真實使用情境",
            "每個情境的使用者目標",
            "輸入 / 系統處理 / 輸出",
            "至少 5 句 User Story"
        ],
        "template": """你現在是一位{role}。

請根據以下專案背景、角色分析與痛點，設計這個 AI 報價助理系統的使用情境與 User Story。

請輸出：
1. 3 到 5 個真實使用情境
2. 每個情境中的使用者目標
3. 每個情境的輸入、系統處理、輸出
4. 至少 5 句 User Story，格式為：
   身為【角色】，我希望【做某件事】，以便【達成某目標】。

要求：
- 情境要貼近企業真實流程
- 不要寫得太空泛
- 請用繁體中文
- 請使用 Markdown 小標與條列格式輸出

資料如下：
{context}
"""
    },
    {
        "id": "stage5_goals",
        "title": "5. 專案目標與非目標",
        "role": "專案規劃顧問",
        "instruction": "定義系統要做什麼與不做什麼。",
        "output_requirements": [
            "3 到 5 項核心目標",
            "至少 3 項非目標範圍"
        ],
        "template": """你現在是一位{role}。

請根據以下企業案例與使用情境，幫我整理：
1. 本專案的核心目標
2. 本專案第一階段不處理的非目標範圍

要求：
- 核心目標請列出 3 到 5 項
- 非目標至少列出 3 項
- 每一項都要具體，不要寫成口號
- 要能作為需求規格書內容
- 請用繁體中文
- 請用條列輸出

資料如下：
{context}
"""
    },
    {
        "id": "stage6_functional",
        "title": "6. 功能需求",
        "role": "企業系統分析師",
        "instruction": "將情境轉成可開發、可驗收的功能需求。",
        "output_requirements": [
            "功能編號",
            "功能名稱",
            "功能說明",
            "輸入",
            "輸出",
            "驗收方式"
        ],
        "template": """你現在是一位{role}。

請根據以下專案背景、使用情境與目標，幫我撰寫需求規格書中的功能需求。

請輸出：
1. 功能需求清單
2. 每個功能需求的編號（例如 F1, F2, F3）
3. 每個功能的功能名稱
4. 每個功能的功能說明
5. 每個功能的輸入
6. 每個功能的輸出
7. 每個功能的驗收方式

要求：
- 以「系統應能夠……」的方式撰寫
- 內容要具體、可驗收
- 至少列出 5 到 8 項功能需求
- 請用 Markdown 表格輸出
- 使用繁體中文

資料如下：
{context}
"""
    },
    {
        "id": "stage7_nonfunctional",
        "title": "7. 非功能需求",
        "role": "資深軟體需求分析師",
        "instruction": "定義效能、可用性、安全、追溯性與可維護性。",
        "output_requirements": [
            "效能需求",
            "可用性需求",
            "安全與權限需求",
            "可追溯性需求",
            "可維護性與擴充性需求"
        ],
        "template": """你現在是一位{role}。

請根據以下專案內容，幫我撰寫此系統的非功能需求。

請至少涵蓋以下面向：
1. 效能需求
2. 可用性需求
3. 安全與權限需求
4. 可追溯性需求
5. 可維護性與擴充性需求

要求：
- 每一類至少列出 2 到 4 點
- 要用正式、具體、可檢查的描述
- 適合放進企業需求規格書
- 請使用繁體中文
- 用分段條列方式輸出

專案資料如下：
{context}
"""
    },
    {
        "id": "stage8_data",
        "title": "8. 資料需求",
        "role": "資料分析與系統設計顧問",
        "instruction": "整理資料來源、欄位與資料結構。",
        "output_requirements": [
            "主要資料來源",
            "用途",
            "重要資料欄位",
            "是否必填",
            "結構化 / 非結構化"
        ],
        "template": """你現在是一位{role}。

請根據以下專案內容，幫我整理這個 AI 報價助理系統的資料需求。

請輸出：
1. 主要資料來源
2. 每個資料來源的用途
3. 重要資料欄位清單
4. 欄位名稱、欄位說明、是否必填
5. 哪些資料是結構化、哪些是非結構化

要求：
- 內容要以需求規格書角度撰寫
- 用表格輸出
- 使用繁體中文

資料如下：
{context}
"""
    },
    {
        "id": "stage9_boundary",
        "title": "9. 系統邊界與限制",
        "role": "企業 IT 專案顧問",
        "instruction": "收斂範圍，界定第一版系統邊界。",
        "output_requirements": [
            "系統邊界",
            "第一階段不處理的範圍",
            "已知限制",
            "需要人工介入的地方"
        ],
        "template": """你現在是一位{role}。

請根據以下專案內容，幫我整理這個系統的：
1. 系統邊界
2. 第一階段不處理的範圍
3. 已知限制
4. 需要人工介入的地方

要求：
- 請用正式、明確的語氣
- 請區分「系統不做」與「系統目前做不到」
- 請使用繁體中文
- 用條列方式輸出

資料如下：
{context}
"""
    },
    {
        "id": "stage10_acceptance",
        "title": "10. 驗收標準與 KPI",
        "role": "專案驗收與品質管理顧問",
        "instruction": "讓需求變成可驗收、可評估的規格。",
        "output_requirements": [
            "功能驗收標準",
            "專案成功指標（KPI）"
        ],
        "template": """你現在是一位{role}。

請根據以下需求規格內容，幫我整理：
1. 功能驗收標準
2. 專案成功指標（KPI）

要求：
- 功能驗收標準要能對應到功能需求
- KPI 要具體，例如時間縮短、遺漏率下降、使用效率提升
- 不要寫太空泛的文字
- 請用表格輸出
- 使用繁體中文

資料如下：
{context}
"""
    }
]

FINAL_TEMPLATE = """你現在是一位資深企業系統分析師。

我已經完成一份 AI 報價助理系統需求規格書的各段內容，請幫我整併成一份正式、完整、條理清楚的需求規格書。

請依照以下章節排序整合：
1. 專案背景
2. 專案動機
3. 問題描述
4. 專案目標
5. 非目標範圍
6. 利害關係人與使用者角色
7. 使用情境與 User Story
8. 現況流程（As-Is）
9. 未來流程（To-Be）
10. 功能需求
11. 非功能需求
12. 資料需求
13. 系統邊界與限制
14. 驗收標準與 KPI
15. 風險與假設

要求：
- 使用正式規格書語氣
- 保持條理清楚
- 避免重複
- 若內容有明顯矛盾，請指出並提出修正建議
- 請使用繁體中文
- 請輸出 Markdown 格式

各段資料如下：
{context}
"""


In [ ]:

# ===== Ollama 呼叫工具 =====
import requests, json, textwrap, re, os, time
from datetime import datetime

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"

def ollama_chat(model: str, system_prompt: str, user_prompt: str, temperature: float = 0.2, timeout: int = 600):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    return data["message"]["content"]

def build_context(case_story: str, prior_outputs: dict, include_prior: bool = True):
    parts = []
    if case_story.strip():
        parts.append("# 原始企業案例故事\n" + case_story.strip())
    if include_prior and prior_outputs:
        for s in STAGE_DEFS:
            sid = s["id"]
            if sid in prior_outputs and str(prior_outputs[sid]).strip():
                parts.append(f"# {s['title']} 的既有結果\n" + prior_outputs[sid].strip())
    return "\n\n".join(parts).strip()

def build_stage_prompt(stage_id: str, case_story: str, prior_outputs: dict, include_prior: bool = True):
    stage = next(s for s in STAGE_DEFS if s["id"] == stage_id)
    context = build_context(case_story, prior_outputs, include_prior=include_prior)
    return stage["template"].format(role=stage["role"], context=context)

def export_markdown(case_title: str, case_story: str, outputs: dict):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = [
        f"# {case_title or 'AI 報價助理系統需求規格書'}",
        "",
        f"- 匯出時間：{now}",
        "",
        "## 原始案例故事",
        case_story.strip() or "（未填）",
        ""
    ]
    for s in STAGE_DEFS:
        content = outputs.get(s["id"], "").strip()
        lines.append(f"## {s['title']}")
        lines.append(content if content else "（尚未完成）")
        lines.append("")
    return "\n".join(lines).strip()


In [ ]:

# ===== Gradio 互動介面 =====
import gradio as gr

DEFAULT_CASE = """某家中大型線纜製造公司經常收到來自 Email、LINE、PDF 規格書與電話的客戶詢價需求。
業務人員需要手動整理規格、檢查資訊是否完整、查找歷史報價案例，並與工程、採購、製造部門來回確認。
新手業務報價很慢，老手業務高度依賴經驗，歷史資料散落在 Excel、信件與 ERP 中，導致報價效率低、標準不一致，且知識難以傳承。
公司希望建立一套 AI 報價助理系統，先協助完成需求整理、缺漏檢查、相似案例檢索，以及初步報價建議摘要。"""

def init_outputs():
    return {s["id"]: "" for s in STAGE_DEFS}

def stage_info(stage_title):
    stage = next((s for s in STAGE_DEFS if s["title"] == stage_title), None)
    if not stage:
        return "", "", ""
    req = "\n".join([f"- {x}" for x in stage["output_requirements"]])
    return stage["instruction"], req, stage["role"]

def generate_one(stage_title, case_title, case_story, model_name, temperature, include_prior, outputs_state, custom_note):
    stage = next((s for s in STAGE_DEFS if s["title"] == stage_title), None)
    if stage is None:
        raise gr.Error("找不到對應階段。")
    outputs_state = outputs_state or init_outputs()
    prompt = build_stage_prompt(stage["id"], case_story, outputs_state, include_prior=include_prior)
    if custom_note and custom_note.strip():
        prompt += "\n\n# 額外老師/學生補充要求\n" + custom_note.strip()
    system_prompt = f"你是一位{stage['role']}，擅長將模糊的企業需求整理成正式、可執行、可驗收的需求規格內容。"
    result = ollama_chat(model_name, system_prompt, prompt, temperature=temperature)
    outputs_state[stage["id"]] = result
    preview = export_markdown(case_title, case_story, outputs_state)
    return result, outputs_state, preview

def revise_current(stage_title, current_text, model_name, revise_instruction):
    if not current_text.strip():
        raise gr.Error("目前階段還沒有內容可修正。")
    revise_prompt = f"""請根據以下修正要求，改寫這段需求規格內容。
要求：
- 保留原意
- 改成更正式、具體、可執行、可驗收
- 使用繁體中文
- 請直接輸出修正版 Markdown

修正要求：
{revise_instruction}

原內容如下：
{current_text}
"""
    system_prompt = "你是一位資深企業系統分析師，負責把草稿改寫成正式規格書內容。"
    result = ollama_chat(model_name, system_prompt, revise_prompt, temperature=0.2)
    return result

def save_current(stage_title, edited_text, outputs_state, case_title, case_story):
    stage = next((s for s in STAGE_DEFS if s["title"] == stage_title), None)
    if stage is None:
        raise gr.Error("找不到對應階段。")
    outputs_state = outputs_state or init_outputs()
    outputs_state[stage["id"]] = edited_text
    preview = export_markdown(case_title, case_story, outputs_state)
    return outputs_state, preview

def compile_final(case_title, case_story, model_name, outputs_state):
    outputs_state = outputs_state or init_outputs()
    context_parts = []
    for s in STAGE_DEFS:
        content = outputs_state.get(s["id"], "").strip()
        if content:
            context_parts.append(f"# {s['title']}\n{content}")
    if not context_parts:
        raise gr.Error("目前沒有可整併的內容。")
    prompt = FINAL_TEMPLATE.format(context="\n\n".join(context_parts))
    final_doc = ollama_chat(
        model_name,
        "你是一位資深企業系統分析師，擅長整併多段需求內容成正式需求規格書。",
        prompt,
        temperature=0.2,
        timeout=900
    )
    outputs_state = outputs_state or init_outputs()
    outputs_state["final_document"] = final_doc
    return final_doc, outputs_state

def download_markdown(case_title, case_story, outputs_state):
    outputs_state = outputs_state or init_outputs()
    md_text = export_markdown(case_title, case_story, outputs_state)
    fname = f"{(case_title or 'quotation_spec').replace(' ', '_')}.md"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(md_text)
    return fname

with gr.Blocks(title="AI 報價助理｜需求規格書互動教學") as demo:
    gr.Markdown("""
    # AI 報價助理
    """)

    outputs_state = gr.State(init_outputs())

    with gr.Row():
        case_title = gr.Textbox(label="專案名稱", value="AI 報價助理系統需求規格書")
        model_name = gr.Textbox(label="Ollama 模型名稱", value=MODEL_NAME)
        temperature = gr.Slider(0, 1, value=0.2, step=0.05, label="Temperature")

    case_story = gr.Textbox(
        label="企業案例故事 / 教師提供情境",
        value=DEFAULT_CASE,
        lines=8
    )

    with gr.Row():
        stage_dropdown = gr.Dropdown(
            choices=[s["title"] for s in STAGE_DEFS],
            value=STAGE_DEFS[0]["title"],
            label="目前階段"
        )
        include_prior = gr.Checkbox(value=True, label="生成時帶入前面階段的既有結果")

    with gr.Row():
        stage_instruction = gr.Textbox(label="本階段教學目的", interactive=False)
        stage_requirements = gr.Textbox(label="本階段預期輸出", interactive=False)
        stage_role = gr.Textbox(label="模型角色", interactive=False)

    custom_note = gr.Textbox(
        label="額外補充要求（可選）",
        placeholder="例如：請更貼近製造業情境、請加入資深業務與新手業務的差異、請改成表格...",
        lines=3
    )

    with gr.Row():
        generate_btn = gr.Button("產生本階段內容", variant="primary")
        save_btn = gr.Button("保存目前編輯結果")
        compile_btn = gr.Button("整併成完整規格書")

    current_output = gr.Textbox(label="本階段輸出（可直接修改）", lines=20)

    with gr.Accordion("修正版助手", open=False):
        revise_instruction = gr.Textbox(
            label="修正要求",
            value="請改寫得更具體、正式、可驗收，避免空泛敘述。",
            lines=3
        )
        revise_btn = gr.Button("依修正要求重寫目前內容")

    with gr.Accordion("文件預覽與下載", open=True):
        preview_output = gr.Textbox(label="目前累積的 Markdown 文件預覽", lines=20)
        download_btn = gr.Button("輸出目前累積內容為 Markdown 檔")
        file_output = gr.File(label="下載 Markdown 檔")

    final_output = gr.Textbox(label="整併後的正式需求規格書", lines=24)

    stage_dropdown.change(
        fn=stage_info,
        inputs=stage_dropdown,
        outputs=[stage_instruction, stage_requirements, stage_role]
    )

    generate_btn.click(
        fn=generate_one,
        inputs=[stage_dropdown, case_title, case_story, model_name, temperature, include_prior, outputs_state, custom_note],
        outputs=[current_output, outputs_state, preview_output]
    )

    revise_btn.click(
        fn=revise_current,
        inputs=[stage_dropdown, current_output, model_name, revise_instruction],
        outputs=current_output
    )

    save_btn.click(
        fn=save_current,
        inputs=[stage_dropdown, current_output, outputs_state, case_title, case_story],
        outputs=[outputs_state, preview_output]
    )

    compile_btn.click(
        fn=compile_final,
        inputs=[case_title, case_story, model_name, outputs_state],
        outputs=[final_output, outputs_state]
    )

    download_btn.click(
        fn=download_markdown,
        inputs=[case_title, case_story, outputs_state],
        outputs=file_output
    )

    demo.load(
        fn=lambda: stage_info(STAGE_DEFS[0]["title"]),
        inputs=None,
        outputs=[stage_instruction, stage_requirements, stage_role]
    )

demo


Gradio Blocks instance: 7 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.dropdown.Dropdown object at 0x7cb3a08cc080>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7cb3a08b6030>
 |-<gradio.components.textbox.Textbox object at 0x7cb3a08cf320>
 |-<gradio.components.textbox.Textbox object at 0x7cb3a1d57680>
fn_index=1
 inputs:
 |-<gradio.components.dropdown.Dropdown object at 0x7cb3a08cc080>
 |-<gradio.components.textbox.Textbox object at 0x7cb3a0a446b0>
 |-<gradio.components.textbox.Textbox object at 0x7cb3a0a88200>
 |-<gradio.components.textbox.Textbox object at 0x7cb3a08cc620>
 |-<gradio.components.slider.Slider object at 0x7cb3a0929520>
 |-<gradio.components.checkbox.Checkbox object at 0x7cb3a0902270>
 |-<gradio.components.state.State object at 0x7cb3a0a462d0>
 |-<gradio.components.textbox.Textbox object at 0x7cb3a1c0da30>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7cb3a0a2c770>
 |-<gradio.component

In [ ]:

# 啟動 Gradio 前端介面
demo.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://61ec7e28a302c1b6b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 想一想
1. 哪些內容雖然寫得漂亮，但不夠可驗收  
2. 哪些需求其實是流程問題，不一定要交給 LLM  
3. 哪些地方要保留 human-in-the-loop  
4. 哪些地方要加上 log、權限、版本與可追溯性  
